In [ ]:
import re
from google.colab import files

## I choose a text.txt file with a long wsp conversation (+5 years)
uploaded = files.upload()

def parse_whatsapp(file_path):
    pattern = r'(\d+/\d+/\d+)\s+(\d+:\d+)\s+-\s+(.+?):\s+(.+)'
    messages = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            match = re.match(pattern, line)
            if match:
                messages.append({
                    'timestamp': f"{match.group(1)} {match.group(2)}",
                    'sender': match.group(3).strip(),
                    'text': match.group(4).strip()
                })
    return messages

def build_pairs(messages, input_sender="Javi", output_sender="Lafalce Mateo"):
    EXCLUDED = ["<multimedia omitido>", "<media omitted>", "audio omitido", "imagen omitida", "video omitido"]

    def is_valid(text):
        return not any(excl in text.lower() for excl in EXCLUDED)

    grouped = []
    i = 0
    while i < len(messages):
        current_sender = messages[i]['sender']
        group_texts = []
        while i < len(messages) and messages[i]['sender'] == current_sender:
            if is_valid(messages[i]['text']):
                group_texts.append(messages[i]['text'])
            i += 1
        if group_texts:
            grouped.append({
                'sender': current_sender,
                'text': ' '.join(group_texts)
            })

    pairs = []
    for i in range(len(grouped) - 1):
        if grouped[i]['sender'] == input_sender and grouped[i+1]['sender'] == output_sender:
            pairs.append({
                'input': grouped[i]['text'],
                'output': grouped[i+1]['text']
            })

    return pairs

file_path = "text.txt"
messages = parse_whatsapp(file_path)

pairs = build_pairs(messages)


In [ ]:
from google.colab import files
uploaded = files.upload()

file_path = list(uploaded.keys())[0]
print(f"File: {file_path}")

messages = parse_whatsapp(file_path)
pairs = build_pairs(messages)
print(f"Pairs: {len(pairs)}")

corpus_path = "corpus.txt"
with open(corpus_path, "w", encoding="utf-8") as f:
    for p in pairs:
        f.write(p['input'] + "\n")
        f.write(p['output'] + "\n")


In [ ]:
import torch
import torch.nn as nn
import math

# ---------- Hiperparámetros ----------
VOCAB_SIZE   = tokenizer.get_vocab_size()
D_MODEL      = 256
N_HEADS      = 4
N_LAYERS     = 4
D_FF         = 1024
MAX_LEN      = 64          # mismo max_len que encode_pair()
DROPOUT      = 0.1

print(f"Vocab size: {VOCAB_SIZE}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# ---------- Bloques ----------
class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.ff   = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
        )
        self.ln1  = nn.LayerNorm(d_model)
        self.ln2  = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        normed   = self.ln1(x)
        attn_out, _ = self.attn(normed, normed, normed, attn_mask=mask)
        x = x + self.drop(attn_out)
        x = x + self.drop(self.ff(self.ln2(x)))
        return x

class MiniGPT(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, n_layers, d_ff, max_len, dropout=0.1):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.drop    = nn.Dropout(dropout)
        self.blocks  = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)
        ])
        self.ln_f    = nn.LayerNorm(d_model)
        self.head    = nn.Linear(d_model, vocab_size, bias=False)

        # Weight tying: comparte pesos entre embedding y capa de salida
        self.head.weight = self.tok_emb.weight

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, mean=0.0, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, idx):
        B, T = idx.shape
        tok  = self.tok_emb(idx)
        pos  = self.pos_emb(torch.arange(T, device=idx.device))
        x    = self.drop(tok + pos)

        # Causal mask (triangular superior)
        mask = torch.triu(torch.ones(T, T, device=idx.device), diagonal=1).bool()
        for block in self.blocks:
            x = block(x, mask)

        x = self.ln_f(x)
        return self.head(x)

    @torch.no_grad()
    def generate(self, idx, max_new_tokens=50, temperature=0.8, top_k=40):
        """Genera tokens autoregressivamente dado un prompt."""
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -MAX_LEN:]          # recortar si supera max_len
            logits   = self(idx_cond)              # (B, T, vocab)
            logits   = logits[:, -1, :] / temperature

            # Top-k sampling
            if top_k is not None:
                values, _ = torch.topk(logits, top_k)
                logits[logits < values[:, [-1]]] = float('-inf')

            probs    = torch.softmax(logits, dim=-1)
            next_tok = torch.multinomial(probs, num_samples=1)
            idx      = torch.cat([idx, next_tok], dim=1)

            if next_tok.item() == eos_id:
                break
        return idx

model = MiniGPT(
    vocab_size = VOCAB_SIZE,
    d_model    = D_MODEL,
    n_heads    = N_HEADS,
    n_layers   = N_LAYERS,
    d_ff       = D_FF,
    max_len    = MAX_LEN,
    dropout    = DROPOUT,
).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}  (~{total_params/1e6:.1f}M)")

dummy = torch.randint(0, VOCAB_SIZE, (2, MAX_LEN)).to(device)
out   = model(dummy)
print(f"Input shape:  {dummy.shape}")
print(f"Output shape: {out.shape}")

prompt_text  = pairs[0]['input']
prompt_ids   = [sos_id] + tokenizer.encode(prompt_text).ids
prompt_tensor = torch.tensor([prompt_ids], dtype=torch.long).to(device)

generated    = model.generate(prompt_tensor, max_new_tokens=40)
decoded      = tokenizer.decode(generated[0].tolist())

In [ ]:
def chat(text, max_new_tokens=64, temperature=0.8):
    model.eval()
    with torch.no_grad():
        inp_ids, _ = encode_pair({'input': text, 'output': ''}, max_len=MAX_LEN // 2)
        x = torch.tensor(inp_ids, dtype=torch.long).unsqueeze(0).to(device)  # (1, T)

        generated = list(inp_ids)

        for _ in range(max_new_tokens):
            x_input = torch.tensor(generated[-(MAX_LEN-1):], dtype=torch.long).unsqueeze(0).to(device)
            logits = model(x_input)          # (1, T, vocab)
            next_logits = logits[0, -1, :]

            next_logits = next_logits / temperature
            probs = torch.softmax(next_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1).item()

            generated.append(next_token)

            if next_token == eos_id:
                break

        response_ids = generated[len(inp_ids):]
        return tokenizer.decode(response_ids)

In [ ]:
from torch.utils.data import Dataset, DataLoader

MAX_LEN = 128

model = MiniGPT(
    vocab_size = VOCAB_SIZE,
    d_model    = D_MODEL,
    n_heads    = N_HEADS,
    n_layers   = N_LAYERS,
    d_ff       = D_FF,
    max_len    = MAX_LEN,
    dropout    = DROPOUT,
).to(device)

print(f"MAX_LEN={MAX_LEN}")
total_params = sum(p.numel() for p in model.parameters())
print(f"Total Parameters: {total_params:,} (~{total_params/1e6:.1f}M)")

class ChatDataset(Dataset):
    def __init__(self, pairs, tokenizer, max_len=128):
        self.samples = []
        self.max_len = max_len

        for p in pairs:
            inp_ids, out_ids = encode_pair(p, max_len=max_len // 2)
            sequence = inp_ids + out_ids
            self.samples.append(torch.tensor(sequence, dtype=torch.long))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        seq = self.samples[idx]
        return seq[:-1], seq[1:]

dataset    = ChatDataset(pairs, tokenizer, max_len=MAX_LEN)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)
print(f"Dataset: {len(dataset)} | Batches: {len(dataloader)}")
print(f"Shape: {dataset[0][0].shape}")  # debería ser (127,)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)
criterion = nn.CrossEntropyLoss(ignore_index=pad_id)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)

EPOCHS = 50

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0

    for x, y in dataloader:
        x, y = x.to(device), y.to(device)

        logits = model(x)                          # (B, T, vocab)
        loss = criterion(
            logits.view(-1, VOCAB_SIZE),           # (B*T, vocab)
            y.view(-1)                             # (B*T,)
        )

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()

    scheduler.step()
    avg_loss = total_loss / len(dataloader)

    if epoch % 5 == 0:
        print(f"Epoch {epoch:3d}/{EPOCHS} | Loss: {avg_loss:.4f} | LR: {scheduler.get_last_lr()[0]:.2e}")
        model.eval()
        sample = chat(pairs[0]['input'])
        print(f"  >> Input:    {pairs[0]['input']}")
        print(f"  >> Target: {pairs[0]['output']}")
        print(f"  >> Model:   {sample}\n")

torch.save({
    'model_state': model.state_dict(),
    'optimizer_state': optimizer.state_dict(),
    'epoch': EPOCHS,
    'vocab_size': VOCAB_SIZE,
}, "minigpt_chat.pt")

from google.colab import files
files.download("minigpt_chat.pt")

In [ ]:
while True:
    user_input = input("You: ").strip()

    if user_input.lower() in ["salir", "exit", "quit"]:
        break

    if not user_input:
        continue

    respuesta = chat(user_input)
    print(f"Lafalce Mateo: {respuesta}\n")